# RAG System Prototype for Course Material Question Answering

This notebook demonstrates a small scale implementation of a Retrieval-Augmented Generation (RAG) system. The system processes and indexes sample course material, retrieves relevant chunks based on a user query, builds a prompt, and finally generating an answer using a language model (LLM).

In [2]:
# Uncomment and run the following cell to install required packages if needed
# !pip install sentence-transformers faiss-cpu openai
# !pip install transformers accelerate torch

In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

huggingface_models = [
    "gpt2",                                # Small but fast
    "EleutherAI/gpt-neo-125M",             # Open GPT-3-style model
    "tiiuae/falcon-rw-1b"                  # Small Falcon model
]


In [16]:
from openai import OpenAI

openAiKeyFile = open('openaikey.txt', 'r', encoding='utf-8')
OpenAI_API_KEY = openAiKeyFile.readline().strip()
openAiKeyFile.close()
openai_models = [
    "gpt-4",
    "gpt-4-turbo",
    "gpt-3.5-turbo",
    "gpt-3.5-turbo-16k"
]
LLM_Model = openai_models[0]

# Define the LlmAgent class
class LlmAgent:
    def __init__(self, name, key, model, temperature):
        self.name = name
        self.temperature = temperature
        self.model = model
        self.client = OpenAI(api_key=key)
        self.messages = []

    def initializeAgent(self, system_prompt):
        self.messages.append({"role": "system", "content": system_prompt})

    def sendMessage(self, userinput):
        self.messages.append({"role": "user", "content": userinput})

    def getResponse(self):
        response = self.client.chat.completions.create(model=self.model, messages=self.messages)
        response_message = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": response_message})
        return response_message

# Define a function to create an agent
def create_agent(role_name, description, key, model=LLM_Model, temperature=1.0):
    agent = LlmAgent(name=role_name, key=key, model=model, temperature=temperature)
    system_prompt = f"""
    You are the {role_name}. {description}
    """
    agent.initializeAgent(system_prompt)
    return agent
    

In [ ]:
import numpy as np
import nltk
from sentence_transformers import SentenceTransformer
import faiss

#############################################
# 1. Define Functions for the RAG Prototype #
#############################################

# download nltk data
nltk.download('all')

# take in text with punctuation and split into chunks
def chunk_text(text, chunk_size=5, overlap=3):
    sentences = nltk.sent_tokenize(text)
    chunks = []
    start = 0
    while start < len(sentences):
        chunk = sentences[start: start + chunk_size]
        chunks.append(' '.join(chunk))
        start += (chunk_size - overlap)
    return chunks

# Load a pre-trained Sentence Transformer model for embedding
model = SentenceTransformer('all-MiniLM-L6-v2')

def embed_texts(texts):
    """
    Computes vector embeddings for a list of texts using the pre-trained model.
    """
    embeddings = model.encode(texts)
    return embeddings

def build_faiss_index(embeddings):
    """
    Builds a FAISS index from the given embeddings for efficient similarity search.
    """
    dim = embeddings.shape[1]
    #index = faiss.IndexFlatL2(dim)
    index = faiss.IndexFlatIP(dim) # inner product
    faiss.normalize_L2(embeddings) # new =)
    index.add(embeddings.astype('float32'))
    #index.add(embeddings)
    return index

def keyword_score(query, chunk):
    query = query.split() # ["What", "is", "y"]
    chunk = chunk.split() # ["What", "is", "y"]

    query_words = {word.strip('.,!?').lower() for word in query}
    chunk_words = {word.strip('.,!?').lower() for word in chunk}
    return len(query_words.intersection(chunk_words))

def retrieve(query, chunks, index, k=3):
    """
    Retrieves the top-k most relevant text chunks for a given query.
    """
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding) # new =)
    D, I = index.search(np.array(query_embedding).astype('float32'), 2 * k)
    # print(I[0])
    # print(len(chunks))
    retrieved_chunks = [chunks[i] for i in I[0]]

    retrieved_chunks.sort(key = lambda chunk: keyword_score(query, chunk), reverse = True)

    return retrieved_chunks[:k]
    

    #return retrieved_chunks

def build_prompt(query, retrieved_chunks):
    """
    Combines the user query with retrieved context chunks to build a prompt for the LLM.
    """
    context_chunks = "\n\n---------\n\n".join(retrieved_chunks)

    prompt = f"""You are a helpful teaching assistant answering questions based on course materials.\n\nContext: {context_chunks}\n\nQuestion: {query}\n\nAnswer:"""
    return prompt



In [44]:
#########################################
# 2. Process and Index Course Material  #
#########################################
import os
import numpy as np
from IPython.display import display, Latex

# 2.0: Helper to load all .txt files from given folders
def load_all_texts(dirs, extensions=(".txt",)):
    docs = {}
    for d in dirs:
        for fname in os.listdir(d):
            if fname.lower().endswith(extensions):
                path = os.path.join(d, fname)
                with open(path, "r", encoding="utf-8") as f:
                    docs[fname] = f.read()
    return docs

# 2.1: Load documents
directories = ["lecture_notes", "transcripts"]
documents = load_all_texts(directories)

# 2.2: Chunk each document and collect all chunks
all_chunks = []
for fname, text in documents.items():
    print(f"\n--- Processing '{fname}' ---")
    # (Optionally) Render the raw document once
    #display(Latex(r"\section*{" + fname.replace("_", r"\_") + r"}"))
    #display(Latex(text))
    
    # Chunk it
    chunks = chunk_text(text, chunk_size=50, overlap=10)
    for c in chunks:
        # (Optionally) render each chunk in LaTeX to inspect
        #display(Latex(c))
        print("-" * 50)
    all_chunks.extend(chunks)

print(f"\nTotal chunks created: {len(all_chunks)}")

# 2.3: Embed all chunks
embeddings = embed_texts(all_chunks)
embeddings_np = np.array(embeddings).astype("float32")

# 2.4: Build the FAISS index
index = build_faiss_index(embeddings_np)

print("✅ All documents processed, embedded, and indexed.")



--- Processing 'Hands_On_ML_Rewritten.txt' ---
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------

--- Processing 'Kernel_SVMs_Rewritten.txt' ---
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
-------------------------------------

In [18]:
# #########################################
# # 2. Process and Index Course Material  #
# #########################################
# from IPython.display import display, Latex

# # Read course material from a file
# with open("course_material.txt", "r", encoding="utf-8") as f:
#     course_material = f.read()
    
# # Print to confirm
# print("Loaded course material:\n")
# display(Latex(course_material))

# # Step 2.1: Chunk the course material
# chunks = chunk_text(course_material, chunk_size=50, overlap=10)
# print("Chunks:\n")
# for c in chunks:
#     display(Latex(c))
#     print("-"*50)

# # Step 2.2: Embed the text chunks
# embeddings = embed_texts(chunks)
# embeddings_np = np.array(embeddings).astype('float32')

# # Step 2.3: Build the FAISS index
# index = build_faiss_index(embeddings_np)

In [54]:
##########################################
# 3. Retrieval, Prompt Building, and LLM #
##########################################

# Step 3.1: Define a sample query
#query = "What programming paradigms does Python support?"
query = "Why does the SVD of a matrix reveal its rank?"

# Step 3.2: Retrieve the top-k relevant chunks from the course material
retrieved_chunks = retrieve(query, all_chunks, index, k=5) ### updated from repo, k should be 10
print("\nRetrieved Chunks:\n")
for chunk in retrieved_chunks:
    display(Latex(chunk))
    print("-"*50)

# Step 3.3: Build a prompt combining the query and retrieved context
prompt = build_prompt(query, retrieved_chunks)
print("\nPrompt for LLM:\n")
print(prompt)
# display(Latex(prompt))


Retrieved Chunks:



<IPython.core.display.Latex object>

--------------------------------------------------


<IPython.core.display.Latex object>

--------------------------------------------------


...

--------------------------------------------------


<IPython.core.display.Latex object>

--------------------------------------------------


<IPython.core.display.Latex object>

--------------------------------------------------

Prompt for LLM:

You are a helpful teaching assistant answering questions based on course materials.

Context: of the Matrix and that's exactly equal to the number of nonzero singular values that's really the almost numerically the best way to determine the rank of a matrix is to actually do an SVD okay so so so let me write it out this means that Sigma 1 is

---------

can actually take for example K = to 1 or k = to 2 okay rank of AK is K and the SVD has the property and remember AK is the sorry I I AK is this Matrix so AK is clearly of rank K it's an M byn Matrix

---------

how the SVD helps us over there I have a question if I made so those uh Sigma values like the singular values like Sigma 1 to Sigma n okay so when we do the the K rank uh so you mentioned that these are like kind of sorted so

---------

of course ordered in decreasing order and then also the corresponding v's and that is sometimes called the K truncated SVD an

In [56]:
# Step 3.4: Generate an answer using OpenAI LLM
the_llm_agent = create_agent(role_name="RAG Agent", description="", key=OpenAI_API_KEY, model=LLM_Model, temperature=0.0)
the_llm_agent.sendMessage(prompt)
answer = the_llm_agent.getResponse()
print("\nBig LLM Answer:\n")
print(answer)
#display(Latex(answer))


Big LLM Answer:

The Singular Value Decomposition (SVD) of a matrix reveals its rank because the rank of a matrix is equal to the number of singular values that are not zero. When we perform SVD on a matrix, it decomposes the matrix into three other matrices. One of these matrices is a diagonal matrix of singular values. The non-zero values in this diagonal matrix indicate the independent rows/columns in the original matrix, thus revealing its rank. Because the singular values are sorted in descending order, we can easily see the most significant dimensions of the matrix. This is also why a SVD can provide an optimal low-rank approximation of the original matrix by considering only the highest singular values.


In [35]:
# Step 3.5: Generate an answer using a smaller LLM

model_name = huggingface_models[1]
tokenizer = AutoTokenizer.from_pretrained(model_name)
smallModel = AutoModelForCausalLM.from_pretrained(model_name)

# Encode input and generate output
inputs = tokenizer(prompt, return_tensors="pt")
outputs = smallModel.generate(**inputs, max_new_tokens=100)

# Decode the output
generated = outputs[0][inputs["input_ids"].shape[1]:]  # Only keep new tokens
response = tokenizer.decode(generated, skip_special_tokens=True)
print("\nSmall LLM Answer:\n")
print(response)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Small LLM Answer:

 Because the rank of a matrix is the sum of its columns. So if you have a matrix A and you want to rank it, you have to sort it by the rank of A. So if you have a matrix A and you want to rank it, you have to sort it by the rank of A. So if you have a matrix A and you want to rank it, you have to sort it by the rank of A. So if you have a matrix A and you want to rank


In [ ]:
import os
import re

def clean_transcript(file_path):
    # Derive the output filename by removing the trailing 'c' before .txt
    base_name = os.path.basename(file_path)
    if not base_name.endswith('c.txt'):
        print(f"Skipped non-matching file: {file_path}")
        return
    
    output_name = base_name[:-5] + '.txt'  # remove 'c' and add '.txt'
    output_path = os.path.join(os.path.dirname(file_path), output_name)

    with open(file_path, 'r', encoding='utf-8') as infile:
        lines = infile.readlines()

    cleaned_lines = []
    i = 0
    while i < len(lines) - 2:
        name1 = lines[i].strip()
        name2 = lines[i + 1].strip()
        time = lines[i + 2].strip()

        if name1 == name2 and re.match(r'^\d{2}:\d{2}:\d{2}$', time):
            i += 3  # skip these three lines
        else:
            cleaned_lines.append(lines[i])
            i += 1

    # Add any remaining lines at the end
    while i < len(lines):
        cleaned_lines.append(lines[i])
        i += 1

    with open(output_path, 'w', encoding='utf-8') as outfile:
        outfile.writelines(cleaned_lines)

    print(f"Cleaned file saved as: {output_path}")


# Example usage
clean_transcript("lecture-13c.txt")


In [ ]:
from IPython.display import display, Latex

# Display equation
display(Math(r'E = mc^2'))

# For text + equation
display(Latex(r'The quadratic formula is: $x = \frac{-b \pm \sqrt{b^2 - 4ac}}{2a}$'))
